# Run 9 — ResNet-50 (ImageNet pretrained) + CBAM + MediaPipe pose fusion

End-to-end Kaggle T4×2 recipe. Swaps the Run 8 from-scratch ResNet-18 backbone
for an ImageNet-pretrained ResNet-50 with CBAM after each of the four stages;
keeps the Run 8 36-d MediaPipe pose vector and fusion MLP head.

Why pretrained ResNet-50: deeper backbone (2048-d features vs 512-d) + ImageNet
transfer should converge faster and to a higher plateau than the Run 8 baseline
if pose features are the bottleneck — and let us decouple "is the CNN under-trained?"
from "do pose features add signal?".

Pre-reqs: `splits/train.csv`, `splits/val.csv`, `splits/stats.json`, and
`splits/pose.parquet` from prior Runs (8). If any missing, run §1b / §2.

Resource notes (T4×2, 16 GB each):
- ResNet-50 @ 384 px batch 48 → OOM on a single T4. Use `--full-size 320`
  + `--batch-size 32` with `--data-parallel`, or `--full-size 384` + batch 16.
- ImageNet pretrained → use **ImageNet mean/std** (`--imagenet-stats`), not
  the dataset stats from `splits/stats.json`.

## 1a. Paths

In [ ]:
import os, sys
COMP_DIR = "/kaggle/input/competitions/state-farm-distracted-driver-detection"
CODE_DIR = "/kaggle/input/driver-distraction-cbam"   # or /kaggle/working/code
WORK     = "/kaggle/working"
RUN      = f"{WORK}/run9"

assert os.path.exists(COMP_DIR + "/driver_imgs_list.csv"), "Competition dataset not attached"
assert os.path.exists(CODE_DIR + "/model_resnet50.py"),     "Run 9 code missing (model_resnet50.py)"
sys.path.insert(0, CODE_DIR)
print("OK. GPU count:", __import__("torch").cuda.device_count())

### If using the GitHub mirror (skip if dataset attached):
```python
!rm -rf /kaggle/working/code
!git clone https://github.com/nxtruoong/DoAnCS231-V2 /kaggle/working/code
CODE_DIR = "/kaggle/working/code"
sys.path.insert(0, CODE_DIR)
```

## 1b. Data prep — only if `splits/stats.json` missing

In [ ]:
import subprocess, os
if not os.path.exists(f"{WORK}/splits/stats.json"):
    subprocess.run([
        "python", f"{CODE_DIR}/data_prep.py",
        "--data-root", COMP_DIR,
        "--out-dir",   f"{WORK}/splits",
        "--batch-size", "64",
        "--num-workers", "4",
    ], check=True)
else:
    print("splits/stats.json already exists, skipping")

## 1c. Install MediaPipe + polars (skip if `pose.parquet` already present)

In [ ]:
import os
if not os.path.exists(f"{WORK}/splits/pose.parquet"):
    !pip install --no-cache-dir mediapipe==0.10.13 polars
    !python -c "from mediapipe.python.solutions import pose as mp_pose; print('OK,', mp_pose.Pose)"
else:
    print("pose.parquet already exists; skipping mediapipe install")
    # polars still needed at training time for the lookup
    !pip install --no-cache-dir polars

## 2. Precompute pose features — one-time, ~18 min (skip if already done)

In [ ]:
import subprocess, os
if not os.path.exists(f"{WORK}/splits/pose.parquet"):
    subprocess.run([
        "python", f"{CODE_DIR}/extract_pose.py",
        "--img-root", f"{COMP_DIR}/imgs/train",
        "--out",      f"{WORK}/splits/pose.parquet",
    ], check=True)
else:
    print("pose.parquet already exists, skipping")

## 3. Smoke test — 2 epochs, ResNet-50 pretrained (~5-7 min)

Verifies the pretrained-load path + ImageNet stats + pose fusion all wire correctly.

In [ ]:
import subprocess
subprocess.run([
    "python", f"{CODE_DIR}/train_twostream.py",
    "--pose-fusion",
    "--backbone", "resnet50",
    "--pretrained",
    "--imagenet-stats",
    "--pose-parquet", f"{WORK}/splits/pose.parquet",
    "--data-root", COMP_DIR,
    "--splits-dir", f"{WORK}/splits",
    "--out-dir",    f"{WORK}/run9_smoke",
    "--epochs", "2",
    "--batch-size", "32",
    "--num-workers", "2",
    "--lr", "0.01",
    "--full-size", "320",
    "--data-parallel",
], check=True)

**Expect:** 2 epochs complete, no OOM. Pretrained ResNet-50 starts hot — val acc
should already be in 0.40-0.70 after ep 2 (much higher than Run 8 ep 2 ~0.20).
If OOM → drop `--batch-size 24` or `--full-size 288`.

## 3b. Restart kernel before full training (recommended)

Cell 2 / smoke test left MediaPipe + worker processes in memory. Restart the
kernel (Run → Restart & Clear Cell Outputs), then re-run Cell 1a only. Skip
1b / 1c / 2 / 3 (outputs already on disk) and go to Cell 4.

## 4. Full Run 9 training (~2-2.5 hr)

**LR note.** Pretrained backbone fine-tunes at lower LR than from-scratch.
`--lr 0.01` (vs Run 8's 0.03) on top of cosine + 2-ep warmup. Stretch: 0.005
if val acc oscillates.

In [ ]:
import subprocess
subprocess.run([
    "python", f"{CODE_DIR}/train_twostream.py",
    "--pose-fusion",
    "--backbone", "resnet50",
    "--pretrained",
    "--imagenet-stats",
    "--pose-parquet", f"{WORK}/splits/pose.parquet",
    "--data-root", COMP_DIR,
    "--splits-dir", f"{WORK}/splits",
    "--out-dir",    RUN,
    "--epochs", "40",
    "--batch-size", "32",
    "--num-workers", "2",
    "--lr", "0.01",
    "--warmup-epochs", "2",
    "--ema-decay", "0.99",
    "--full-size", "320",
    "--label-smoothing", "0.1",
    "--early-stop-patience", "8",
    "--early-stop-min-delta", "0.000",
    "--ckpt-every", "2",
    "--data-parallel",
], check=True)

**Milestones (pretrained → should beat Run 8 early):**

| ep | target ema val acc | Run 8 actual |
|---:|---:|---:|
| 2  | ≥ 0.55 | ~0.20 |
| 10 | ≥ 0.85 | ~0.78 |
| 20 | ≥ 0.89 | ~0.85 |
| 30 | ≥ 0.91 | ~0.87 |
| end | ≥ 0.92 | ~0.88 |

**Save-and-commit recommendation.** Use **Save Version → Save & Run All — Commit**
so the run survives browser disconnects. Total ~2-2.5 hr, well inside the 12-hr cap.

**OOM symptoms / fixes:**
- OOM at batch 32 / 320 px → drop to batch 24, or `--full-size 288`.
- Host RAM > 10 GB by ep 3 → drop `--num-workers 0`, or kill `--data-parallel` (one T4 only).

## 5. Resume if interrupted

In [ ]:
import subprocess, glob
ckpts = sorted(glob.glob(f"{RUN}/ckpt_e*.pt"))
last_ckpt = ckpts[-1] if ckpts else None
print("Resuming from", last_ckpt)
assert last_ckpt is not None

subprocess.run([
    "python", f"{CODE_DIR}/train_twostream.py",
    "--pose-fusion",
    "--backbone", "resnet50",
    "--pretrained",
    "--imagenet-stats",
    "--pose-parquet", f"{WORK}/splits/pose.parquet",
    "--resume", last_ckpt,
    "--data-root", COMP_DIR,
    "--splits-dir", f"{WORK}/splits",
    "--out-dir",    RUN,
    "--epochs", "40",
    "--batch-size", "32",
    "--num-workers", "2",
    "--lr", "0.01",
    "--warmup-epochs", "2",
    "--ema-decay", "0.99",
    "--full-size", "320",
    "--label-smoothing", "0.1",
    "--early-stop-patience", "8",
    "--early-stop-min-delta", "0.000",
    "--ckpt-every", "2",
    "--data-parallel",
], check=True)

## 6. Peek at history

In [ ]:
import json
hist = json.loads(open(f"{RUN}/history.json").read())
print(f"Last epoch: {hist[-1]['epoch']}")
best_idx = max(range(len(hist)), key=lambda i: hist[i]['ema_val_acc'])
print(f"Best EMA:   {hist[best_idx]['ema_val_acc']:.4f} at ep {hist[best_idx]['epoch']}")
print(f"Best raw:   {max(x['val_acc'] for x in hist):.4f}")

## 7. Eval + figures (~5 min)

`eval_twostream.py` reads `backbone` / `pretrained` / `imagenet_stats` from
the saved ckpt args, so no extra flags needed.

In [ ]:
import subprocess
subprocess.run([
    "python", f"{CODE_DIR}/eval_twostream.py",
    "--ckpt",         f"{RUN}/best.pt",
    "--pose-parquet", f"{WORK}/splits/pose.parquet",
    "--data-root",    COMP_DIR,
    "--splits-dir",   f"{WORK}/splits",
    "--out-dir",      f"{RUN}/eval",
    "--history-json", f"{RUN}/history.json",
    "--full-size", "320",
    "--batch-size", "32",
], check=True)

## 8. Compare to Run 6 / 7 / 8

In [ ]:
import json, pandas as pd, os

def metrics(p):
    m = json.load(open(p))
    return {"accuracy": m["accuracy"],
            "macro_f1": m["macro avg"]["f1-score"],
            "weighted_f1": m["weighted avg"]["f1-score"]}

rows = {}
for label, path in [
    ("Run 6 (single stream)",        f"{WORK}/run6/eval/metrics.json"),
    ("Run 7 (two stream)",           f"{WORK}/run7/eval/metrics.json"),
    ("Run 8 (R18 + pose)",           f"{WORK}/run8/eval/metrics.json"),
    ("Run 9 (R50 ImageNet + pose)",  f"{RUN}/eval/metrics.json"),
]:
    if os.path.exists(path):
        rows[label] = metrics(path)
table = pd.DataFrame(rows).T
print(table.to_string())
table.to_csv(f"{WORK}/run9_vs_others.csv")

**Pass criterion for Run 9:** macro F1 ≥ 0.90. Stretch: ≥ 0.93.
If macro F1 < 0.88 → ImageNet transfer didn't help on cabin imagery; report Run 8 as headline.

## 9. Per-class deep dive vs Run 8

In [ ]:
import json, os
m8_path = f"{WORK}/run8/eval/metrics.json"
m9 = json.load(open(f"{RUN}/eval/metrics.json"))
m8 = json.load(open(m8_path)) if os.path.exists(m8_path) else None

print(f"{'class':<25} {'R8 F1':>8} {'R9 F1':>8} {'Δ vs R8':>8}")
for k in m9:
    if k.startswith("c") and "(" in k:
        f9 = m9[k]["f1-score"]
        f8 = m8[k]["f1-score"] if m8 else float("nan")
        d  = (f9 - f8) if m8 else float("nan")
        print(f"{k:<25} {f8:>8.4f} {f9:>8.4f} {d:>+8.4f}")

## 10. Bundle artifacts for download

In [ ]:
import zipfile
from pathlib import Path

OUT = Path(f"{WORK}/artifacts_run9.zip")
OUT.unlink(missing_ok=True)

paths = [
    Path(f"{RUN}/best.pt"),
    Path(f"{RUN}/history.json"),
    *Path(f"{RUN}/eval").iterdir(),
    Path(f"{WORK}/splits/stats.json"),
    Path(f"{WORK}/splits/pose.parquet"),
    Path(f"{WORK}/splits/train.csv"),
    Path(f"{WORK}/splits/val.csv"),
    Path(f"{WORK}/run9_vs_others.csv"),
]
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for p in paths:
        if p.exists():
            z.write(p, p.relative_to(WORK))

print(f"{OUT.name}: {OUT.stat().st_size / 1e6:.1f} MB")
from IPython.display import FileLink, display
display(FileLink(str(OUT)))

Run 9 `best.pt` is ~270 MB (ResNet-50 backbone is ~95 M params + pose head).
For HuggingFace Spaces (<50 MB without LFS) save EMA-only:

```python
import torch
ck = torch.load(f"{RUN}/best.pt", map_location="cpu", weights_only=False)
torch.save({"ema": ck["ema"], "args": ck["args"], "pose_fusion": True},
           f"{RUN}/best_demo.pt")
```

`best_demo.pt` is ~95 MB — still needs LFS for HF Spaces or a split-zip.